# pipe_crowd diagnostics

Plots for one `pipe_crowd` run (one visit / detector), with optional comparison runs
(other `pipe_crowd` collections and `meas_crowd_v2` detector runs).

| Section | What |
| --- | --- |
| 1 | Timing: per step, per detection round, timeline, CPU vs wall, comparison between runs |
| 2 | Data / model / residual (whole detector and a zoom), χ maps and χ distribution |
| 3 | Background: the subtracted background and the sky left in the residual |
| 4 | Source catalogue over data / model / residual |
| 5 | PSF model: kernel, profile, spatial variation, FWHM map, stacked star residuals |
| 6 | Gaia: overlay, completeness, astrometric offsets, photometry vs Gaia G |
| 7 | Source density over the whole detector |
| 8 | Extras: luminosity function, nearest-neighbour separations, run-to-run matching, summary table |
| 9 | Cutouts at the four corners and along the diagonal: data / model / residual per run, and their statistics |

Needs the Rubin stack with `pipe_crowd` set up (for `lsst.pipe.crowd.timing`).
Cells tagged `lsst-load` are the only ones that talk to the Butler; everything after works on the
plain numpy / astropy objects they build (`run`, `gaia`, `others`).

## Parameters

In [ ]:
import os

# --- the run to analyse ------------------------------------------------------
REPO = "dp2_prep"                       # Butler repo path or alias
INSTRUMENT = "LSSTCam"
VISIT = 2025062100433
DETECTOR = 144
RUN_COLLECTION = "u/USERNAME/pipe_crowd/RUN"   # output collection of the pipetask run
INPUT_COLLECTIONS = []                  # extra collections for visit_image / background, if the
                                        # run collection does not chain to its inputs
RUN_NAME = "pipe_crowd"                 # label in the plots

TASK_LABEL = "crowdedField"             # pipeline task label (pipelines/pipeCrowd.yaml)
VISIT_IMAGE = "visit_image"             # input dataset (CrowdedFieldConnections.calexp)
OUTPUT_PREFIX = "pipeCrowd"             # pipeCrowd_src / _model / _residual
BACKGROUND_DATASETS = ["visit_image_background", "calexpBackground"]   # first one found is used

# --- Gaia ----------------------------------------------------------------------
GAIA_REFCAT = "gaia_dr3_20230707"
GAIA_COLLECTIONS = "refcats"
GAIA_CACHE = None                       # e.g. "gaia_cache/v2025062100433_d144.ecsv"
MATCH_RADIUS_PX = 1.0                   # catalogue <-> Gaia / run <-> run match radius

# --- comparisons (all optional) ------------------------------------------------
COMPARE_PIPE_CROWD = {}                 # {"pipe_crowd old": "u/USERNAME/pipe_crowd/OLD"}
MEAS_CROWD_DIRS = {}                    # {"meas_crowd v2": "/path/to/detector_run"}  (stitchTiles output:
                                        #  detector_catalog.fits, detector_mosaic.fits)
EXTRA_TIMING = {}                       # {"meas_crowd v2": "/path/to/timing.csv"}: a table with columns
                                        #  label, seconds and optionally kind, outer (StepTimer records)

# --- plot settings ---------------------------------------------------------------
ZOOM = None                             # (x0, y0, size) in detector px; None = centre of the detector
DENSITY_BIN_PX = 128                    # bin size of the density maps
CUTOUT_SIZE = 300                       # section 9: size of the corner / diagonal cutouts (px)
N_DIAGONAL = 5                          # section 9: cutouts along the diagonal, between the corners
CUTOUT_RESIDUAL_STYLE = "diverging"     # section 9: residual panel "diverging" (white = 0) or "gray"
SKY_BIN_PX = 64                         # bin size of the residual-sky map
FULL_DETECTOR_MAX_PX = 1024             # whole-detector images are block-averaged to about this size
MIN_CENTROID_SEPARATION = 1.0           # CrowdedFieldTaskConfig.minCentroidSeparation, for reference
COLORMAP = "Reds"                       # one-directional maps (density, PSF, background, ...): white -> red
IMAGE_CMAP = "gray"                     # images (data, model, Gaia / run overlays): white stars on black
DIVERGING_CMAP = "RdBu_r"               # residual / χ (second version) and all signed maps (white = 0);
                                        #  its reds and blues are also the colours of lines and markers
FLUX_LABEL = "Flux [nJy]"               # colour-bar label of data / model / residual
FONT_FAMILY = "Open Sans"               # all plot text; the Light weight is downloaded once from
FONT_WEIGHT = "light"                   #  Google Fonts into FONT_DIR if it is not installed
FONT_DIR = os.path.expanduser("~/.cache/pipe_crowd/fonts")
SAVE_DIR = None                         # e.g. "plots": also save every figure as PNG there

## Setup

In [ ]:
%matplotlib inline
import os
import re
import warnings

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, TwoSlopeNorm
from matplotlib.patches import Patch, Rectangle
import matplotlib.patheffects as pe
from cycler import cycler
from astropy.table import Table
from astropy.visualization import ImageNormalize, ZScaleInterval, AsinhStretch
from scipy.spatial import cKDTree

try:
    import seaborn  # noqa: F401  (registers rocket, mako, ... with matplotlib, if COLORMAP is one)
except ImportError:
    pass
SEQ = plt.get_cmap(COLORMAP)
SEQMAP = SEQ.with_extremes(bad="#bdbdbd")   # masked / NaN pixels in grey
DIVERGING = plt.get_cmap(DIVERGING_CMAP).with_extremes(bad="#9a9a9a")  # masked pixels in grey
# Line / marker palette: the reds and blues of the diverging map, so everything matches the residuals.
BLUE, BLUE_L = DIVERGING(0.06), DIVERGING(0.27)          # dark and light blue
RED, RED_L = DIVERGING(0.94), DIVERGING(0.73)            # dark and light red
GRAY = plt.get_cmap(IMAGE_CMAP).with_extremes(bad=BLUE_L)     # masked pixels in light blue
GRAYLOG = plt.get_cmap(IMAGE_CMAP).with_extremes(bad="black", under="black")  # log-scaled PSF images:
                                                                   #  values <= 0 drawn as the faint end


def vcolors(n, *_):
    """n distinct colours alternating between the blue and the red half of the diverging map
    (the near-white middle is skipped)."""
    nb, nr = (n + 1) // 2, n // 2
    blues = [DIVERGING(v) for v in np.linspace(0.0, 0.36, max(nb, 1))][:nb]
    reds = [DIVERGING(v) for v in np.linspace(1.0, 0.64, max(nr, 1))][:nr]
    out = []
    for i in range(n):
        out.append(blues[i // 2] if i % 2 == 0 else reds[i // 2])
    return out


INK = "#2b2b2b"              # neutral reference lines and text


def useFont(family=FONT_FAMILY, weight=FONT_WEIGHT):
    """Make matplotlib use family/weight (e.g. Open Sans Light) for all text.

    Uses an installed copy if matplotlib knows one; otherwise downloads the static font of that
    weight from Google Fonts into FONT_DIR (once) and registers it. Falls back to the default font
    with a message if that fails (e.g. no internet on the node).
    """
    from matplotlib import font_manager as fm
    import urllib.request
    wnum = {"thin": 100, "extra light": 200, "light": 300, "normal": 400, "regular": 400,
            "medium": 500, "semibold": 600, "bold": 700}.get(str(weight).lower(), weight)
    have = lambda: any(f.name == family and f.weight == wnum for f in fm.fontManager.ttflist)
    if not have():
        path = os.path.join(FONT_DIR, f"{family.replace(' ', '')}-{wnum}.ttf")
        try:
            if not os.path.exists(path):
                os.makedirs(FONT_DIR, exist_ok=True)
                url = (f"https://fonts.googleapis.com/css2?family={family.replace(' ', '+')}:wght@{wnum}")
                req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/4.0"})   # -> .ttf links
                css = urllib.request.urlopen(req, timeout=20).read().decode()
                ttf = re.search(r"url\((https://[^)]+\.ttf)\)", css).group(1)
                urllib.request.urlretrieve(ttf, path)
            fm.fontManager.addfont(path)
        except Exception as e:
            print(f"Font {family} {weight} not available ({type(e).__name__}: {e}); using the default font")
            return False
    import logging
    logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # no "findfont" chatter
    plt.rcParams.update({
        # numeric weight: matplotlib's word "light" means 200, but Open Sans Light is 300
        "font.family": family, "font.weight": wnum,
        "axes.titleweight": wnum, "axes.labelweight": wnum, "figure.titleweight": wnum,
        "mathtext.fontset": "custom", "mathtext.rm": f"{family}:weight={wnum}",
        "mathtext.it": f"{family}:italic:weight={wnum}", "mathtext.bf": f"{family}:weight={wnum}",
        "mathtext.sf": f"{family}:weight={wnum}",
    })
    return True

plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 150, "figure.facecolor": "white",
    "figure.constrained_layout.use": True,
    "image.cmap": COLORMAP, "image.origin": "lower", "image.interpolation": "nearest",
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10, "figure.titlesize": 13,
    "axes.edgecolor": "#555555", "axes.linewidth": 0.8, "axes.labelcolor": INK,
    "xtick.color": "#555555", "ytick.color": "#555555", "xtick.labelsize": 9, "ytick.labelsize": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": "#cccccc", "grid.alpha": 0.6, "grid.linewidth": 0.6,
    "legend.frameon": False, "legend.fontsize": 8.5,
    "lines.linewidth": 1.6, "lines.markersize": 4,
    "axes.prop_cycle": cycler(color=vcolors(6)),
})
useFont()
KIND_COLORS = {"setup": "#9a9a9a", "io": BLUE, "compute": RED, "wall": INK, "quantum": BLUE_L}
RUN_COLORS = [RED, BLUE, RED_L, BLUE_L, INK, "#9a9a9a"]     # main run red, first comparison blue
# Markers on colour-mapped images get an outline in the opposite tone, so they show on both the dark
# sky and the bright stars.
LIGHT_ON_DARK = [pe.withStroke(linewidth=3.2, foreground=INK)]
DARK_ON_LIGHT = [pe.withStroke(linewidth=3.2, foreground="white")]
TEXT_STROKE = [pe.withStroke(linewidth=2.2, foreground="black")]
SRC = dict(marker="o", ls="none", mfc="none", mec="white", mew=1.1, ms=11, path_effects=LIGHT_ON_DARK)
# Markers for the grayscale images: red = this run's sources, blue = Gaia (or the comparison run).
# A thin white outline keeps the dark red / blue visible on black sky; on white stars the colour
# itself stands out.
EDGE_WHITE = [pe.withStroke(linewidth=3.0, foreground="white")]
MK_SRC = dict(marker="o", ls="none", mfc="none", mec=RED, mew=1.8, ms=11, path_effects=EDGE_WHITE)      # red ○
MK_PEAK = dict(marker="+", ls="none", color=BLUE, ms=6, mew=1.4, path_effects=EDGE_WHITE)               # blue +
MK_GAIA = dict(marker="+", ls="none", color=BLUE, ms=11, mew=2.2, path_effects=EDGE_WHITE)              # blue +
MK_GAIA_MISSED = dict(marker="D", ls="none", mfc=BLUE, mec="white", ms=10, mew=1.5)                     # blue ◆
MK_BOTH = dict(marker="o", ls="none", mfc="none", mec=RED_L, mew=1.1, ms=10)                            # light red ○
MK_ONLY_A = dict(marker="o", ls="none", mfc="none", mec=RED, ms=13, mew=2.4, path_effects=EDGE_WHITE)   # red ○
MK_ONLY_B = dict(marker="s", ls="none", mfc=BLUE, mec="white", ms=9, mew=1.5)                           # blue ■
MK_SRC_ON_DIVERGING = dict(marker="o", ls="none", mfc="none", mec=INK, mew=0.9, ms=10)    # dark ○
LEGEND_ON_IMAGE = dict(loc="upper right", frameon=True, facecolor="white", framealpha=0.9)
LEAF_KINDS = ("setup", "io", "compute")   # rows that add up; 'wall'/'quantum' contain other rows


def savefig(fig, name):
    """Save a figure to SAVE_DIR (if set)."""
    if SAVE_DIR:
        os.makedirs(SAVE_DIR, exist_ok=True)
        fig.savefig(os.path.join(SAVE_DIR, f"{name}.png"), bbox_inches="tight", dpi=150)


def show(fig, name):
    """Style, save (if SAVE_DIR is set), display and close a figure."""
    for ax in fig.axes:
        if getattr(ax, "_colorbar", None) is not None:
            ax.tick_params(labelsize=8)
            continue
        if ax.images:
            ax.grid(False)
            for sp in ax.spines.values():
                sp.set_visible(True)
        elif ax.has_data() or ax.lines or ax.patches:
            ax.grid(True)
            ax.set_axisbelow(True)
    savefig(fig, name)
    plt.show()
    plt.close(fig)


def nanquiet(func, *args, **kwargs):
    """Call a nan-function without 'all-NaN slice' / 'mean of empty slice' warnings."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return func(*args, **kwargs)


def block(img, f, func=np.nanmean):
    """Bin an image by f x f (the edge remainder is dropped). Returns (binned, used shape)."""
    if f <= 1:
        return img, img.shape
    ny, nx = (img.shape[0] // f) * f, (img.shape[1] // f) * f
    v = img[:ny, :nx].reshape(ny // f, f, nx // f, f)
    return nanquiet(func, v, axis=(1, 3)), (ny, nx)


def extent(x0, y0, shape):
    """imshow extent in detector pixel coordinates (integer values at pixel centres)."""
    return (x0 - 0.5, x0 + shape[1] - 0.5, y0 - 0.5, y0 + shape[0] - 0.5)


def masked(run, key):
    """Image with bad pixels set to NaN (None if the run has no such image)."""
    a = run.get(key)
    if a is None:
        return None
    a = np.array(a, dtype=np.float32)
    if run.get("bad") is not None:
        a[run["bad"]] = np.nan
    return a


def chiImage(run):
    """Residual / sqrt(variance), NaN on bad pixels."""
    if run.get("resid") is None or run.get("var") is None:
        return None
    with np.errstate(divide="ignore", invalid="ignore"):
        chi = masked(run, "resid") / np.sqrt(run["var"])
    chi[~np.isfinite(chi)] = np.nan
    return chi


def fullView(run, img):
    """Block-average a whole-detector image to about FULL_DETECTOR_MAX_PX. Returns (img, extent, f)."""
    f = max(1, int(np.ceil(max(img.shape) / FULL_DETECTOR_MAX_PX)))
    b, used = block(img, f)
    return b, extent(run["x0"], run["y0"], used), f


def cut(run, img, zoom):
    """Cut the zoom box (x0, y0, size) out of a run image. Returns (img, extent) or (None, None)."""
    if img is None:
        return None, None
    zx, zy, s = zoom
    ix0, iy0 = max(zx - run["x0"], 0), max(zy - run["y0"], 0)
    ix1, iy1 = min(zx + s - run["x0"], img.shape[1]), min(zy + s - run["y0"], img.shape[0])
    if ix1 <= ix0 or iy1 <= iy0:
        return None, None
    return img[iy0:iy1, ix0:ix1], (run["x0"] + ix0 - 0.5, run["x0"] + ix1 - 0.5,
                                   run["y0"] + iy0 - 0.5, run["y0"] + iy1 - 0.5)


def imageNorm(img):
    """zscale + asinh normalisation from the finite pixels of an image."""
    good = np.asarray(img)[np.isfinite(img)]
    if good.size == 0:
        return None
    return ImageNormalize(good, interval=ZScaleInterval(), stretch=AsinhStretch(0.1))


def symNorm(img, pct=99.0):
    """Symmetric linear limits at a percentile of |img|."""
    good = np.abs(np.asarray(img)[np.isfinite(img)])
    v = np.percentile(good, pct) if good.size else 1.0
    return dict(vmin=-v, vmax=v)


def inZoom(cat, zoom, pad=0):
    zx, zy, s = zoom
    return ((cat["x"] >= zx - pad) & (cat["x"] < zx + s + pad) &
            (cat["y"] >= zy - pad) & (cat["y"] < zy + s + pad))


def matchXY(xa, ya, xb, yb, radius):
    """One-to-one nearest-neighbour match of a to b within radius (px).

    Returns indices ia, ib of the matched pairs and their separations.
    """
    xa, ya, xb, yb = (np.asarray(v, float) for v in (xa, ya, xb, yb))
    okA = np.flatnonzero(np.isfinite(xa) & np.isfinite(ya))    # NaN positions never match
    okB = np.flatnonzero(np.isfinite(xb) & np.isfinite(yb))
    if len(okA) == 0 or len(okB) == 0:
        return np.zeros(0, int), np.zeros(0, int), np.zeros(0)
    d, j = cKDTree(np.c_[xb[okB], yb[okB]]).query(np.c_[xa[okA], ya[okA]], distance_upper_bound=radius)
    hit = np.isfinite(d)
    ia, ib, d = okA[hit], okB[j[hit]], d[hit]
    # keep the closest a for each b
    order = np.argsort(d)
    ia, ib, d = ia[order], ib[order], d[order]
    _, first = np.unique(ib, return_index=True)
    return ia[first], ib[first], d[first]


def runningMedian(x, y, bins):
    """Median and 16/84 percentiles of y in bins of x."""
    idx = np.digitize(x, bins)
    c = 0.5 * (bins[1:] + bins[:-1])
    med, lo, hi = (np.full(len(c), np.nan) for _ in range(3))
    for i in range(1, len(bins)):
        v = y[(idx == i) & np.isfinite(y)]
        if len(v) >= 5:
            lo[i - 1], med[i - 1], hi[i - 1] = np.percentile(v, [16, 50, 84])
    return c, med, lo, hi


def areaArcmin2(run, nPix):
    return nPix * (run["pixScale"] / 60.0) ** 2


def detectorRange(run):
    ny, nx = run["shape"]
    return (run["x0"] - 0.5, run["x0"] + nx - 0.5), (run["y0"] - 0.5, run["y0"] + ny - 0.5)

### Reading the timing metadata

The same code as `lsst.pipe.crowd.timing.timingTable`, copied in so the notebook works even when the
`pipe_crowd` set up in this kernel predates the timing code. It reads the step timings `pipe_crowd`
stores in its `<label>_metadata` dataset (if the run has them) and the whole-quantum timing and peak
memory that pipetask records for every run.

In [ ]:
# --- copied from python/lsst/pipe/crowd/timing.py ---
TIMING_COLUMNS = {
    "stepLabel": "label",
    "stepKind": "kind",
    "stepOuter": "outer",
    "stepInner": "inner",
    "stepSeconds": "seconds",
    "stepCpuSeconds": "cpuSeconds",
    "stepStartSeconds": "startSeconds",
    "stepNSources": "nSources",
}


def _findStepMetadata(metadata):
    """Yield (task name, metadata) for every entry holding step records."""
    if "stepLabel" in metadata.keys():
        yield "", metadata
    # Full metadata has one top-level entry per task, e.g. "crowdedField",
    # "crowdedField:centroid", plus "quantum" from the executor.
    for name in metadata.keys():
        try:
            sub = metadata[name]
        except Exception:
            continue
        if hasattr(sub, "keys") and "stepLabel" in sub.keys():
            yield name, sub


def timingTable(metadata, **extraColumns):
    """Convert the step records in a task metadata object into a table.

    Also adds rows of kind ``quantum`` for the phases pipetask's executor
    records around the task (prepare, construct task, execute, total), which
    include framework overhead outside `CrowdedFieldTask.runQuantum`, and a
    ``peakRssMB`` column with the process's peak memory at the end of the
    quantum (-1 if not recorded).

    Parameters
    ----------
    metadata : `lsst.pipe.base.TaskMetadata`
        Full task metadata, e.g. ``butler.get("crowdedField_metadata", ...)``.
    **extraColumns
        Constant columns to add (e.g. ``visit=..., detector=...``).

    Returns
    -------
    table : `astropy.table.Table`
    """
    from astropy.table import Table

    rows = []
    for task, md in _findStepMetadata(metadata):
        arrays = {col: list(md.getArray(key)) for key, col in TIMING_COLUMNS.items()}
        for i in range(len(arrays["label"])):
            row = {col: arrays[col][i] for col in TIMING_COLUMNS.values()}
            row["task"] = task
            rows.append(row)

    quantumRows, peakRssMB = _quantumRows(metadata)
    rows.extend(quantumRows)

    columns = ["task"] + list(TIMING_COLUMNS.values())
    dtypes = [str, str, str, int, int, float, float, float, int]
    casts = dict(zip(columns, dtypes))
    table = Table([[casts[c](r[c]) for r in rows] for c in columns], names=columns,
                  dtype=dtypes)
    table["peakRssMB"] = [peakRssMB] * len(table)
    for name, value in extraColumns.items():
        table[name] = [value] * len(table)
    return table


_QUANTUM_PHASES = [
    ("pipetask: prepare quantum", "prep", "init"),
    ("pipetask: construct task", "init", "start"),
    ("pipetask: execute (runQuantum)", "start", "end"),
    ("pipetask: quantum total", "prep", "end"),
]


def _quantumRows(metadata):
    """Whole-quantum timing recorded by the pipetask executor, if present.

    Returns
    -------
    rows : `list` [`dict`]
        One row per phase in ``_QUANTUM_PHASES`` (kind ``quantum``).
    peakRssMB : `float`
        Peak resident memory of the process at the end of the quantum [MB],
        or -1 if not recorded.
    """
    from datetime import datetime
    try:
        q = metadata["quantum"]
    except Exception:
        return [], -1.0

    def get(key):
        try:
            return q[key]
        except Exception:
            return None

    rows = []
    for label, a, b in _QUANTUM_PHASES:
        ta, tb = get(f"{a}Utc"), get(f"{b}Utc")
        if ta is None or tb is None:
            continue
        try:
            seconds = (datetime.fromisoformat(str(tb)) - datetime.fromisoformat(str(ta))).total_seconds()
        except ValueError:
            continue
        ca, cb = get(f"{a}CpuTime"), get(f"{b}CpuTime")
        cpu = float(cb - ca) if ca is not None and cb is not None else -1.0
        rows.append(dict(task="quantum", label=label, kind="quantum", outer=-1, inner=-1,
                         seconds=seconds, cpuSeconds=cpu, startSeconds=-1.0, nSources=-1))
    rss = get("endMaxResidentSetSize")
    return rows, (float(rss) / 1e6 if rss is not None else -1.0)

## Load the run

Builds `run`, a dict of numpy arrays / astropy tables:
`data`, `model`, `resid`, `var` (float32 images), `bad` (bool image), `x0`, `y0`, `shape`,
`cat` (Table: `x`, `y`, `ra`, `dec`, `flux`, `mag`), `zp`, `pixScale` [arcsec/px], `psf` (kernels and FWHM map),
`background` (image or None), `timing` (the step table from `lsst.pipe.crowd.timing`, or None).

In [ ]:
import lsst.geom as geom
from lsst.daf.butler import Butler

BAD_PLANES = ["SAT", "BAD", "EDGE", "CR", "INTRP", "NO_DATA"]
# pipeCrowd_model files written before the filter label was copied onto the model raise a harmless
# "filter label mismatch" warning when read; the pixels are fine.
warnings.filterwarnings("ignore", message=".*filter label mismatch.*")
DATA_ID = dict(instrument=INSTRUMENT, visit=VISIT, detector=DETECTOR)


def badMask(mask):
    bits = 0
    for p in BAD_PLANES:
        try:
            bits |= mask.getPlaneBitMask(p)
        except Exception:
            pass
    return (mask.array & bits) != 0


def psfInfo(psf, bbox, nGrid=3, nFwhm=8):
    """PSF kernels at the centre and on an nGrid x nGrid grid, and an nFwhm x nFwhm FWHM map [px]."""
    def kernel(x, y):
        return psf.computeKernelImage(geom.Point2D(x, y)).array.astype(float)

    def fwhm(x, y):
        return psf.computeShape(geom.Point2D(x, y)).getDeterminantRadius() * 2.0 * np.sqrt(2 * np.log(2))

    c = bbox.getCenter()
    xs = np.linspace(bbox.getMinX(), bbox.getMaxX(), nGrid + 2)[1:-1]
    ys = np.linspace(bbox.getMinY(), bbox.getMaxY(), nGrid + 2)[1:-1]
    fx = np.linspace(bbox.getMinX(), bbox.getMaxX(), nFwhm)
    fy = np.linspace(bbox.getMinY(), bbox.getMaxY(), nFwhm)
    return dict(center=kernel(c.getX(), c.getY()), centerXY=(c.getX(), c.getY()),
                grid=[[kernel(x, y) for x in xs] for y in ys], gridX=xs, gridY=ys,
                fwhmMap=np.array([[fwhm(x, y) for x in fx] for y in fy]), fwhmX=fx, fwhmY=fy,
                fwhmCenter=fwhm(c.getX(), c.getY()))


def catalogTable(src, zp, wcs):
    """pipeCrowd_src -> Table(x, y, ra, dec, flux, mag, xCoarse, yCoarse)."""
    names = src.schema.getNames()
    x = np.asarray(src["centroid_x"] if "centroid_x" in names else src["coarse_centroid_x"], float)
    y = np.asarray(src["centroid_y"] if "centroid_y" in names else src["coarse_centroid_y"], float)
    flux = np.asarray(src["crowd_psfFlux_flux_instFlux"], float)
    with np.errstate(divide="ignore", invalid="ignore"):
        mag = np.where(flux > 0, zp - 2.5 * np.log10(flux), np.nan)
    t = Table(dict(x=x, y=y, flux=flux, mag=mag,
                   xCoarse=np.asarray(src["coarse_centroid_x"], float),
                   yCoarse=np.asarray(src["coarse_centroid_y"], float)))
    try:
        t["ra"] = np.degrees(np.asarray(src["coord_ra"], float))
        t["dec"] = np.degrees(np.asarray(src["coord_dec"], float))
    except Exception:
        ra, dec = wcs.pixelToSkyArray(x, y, degrees=True)
        t["ra"], t["dec"] = ra, dec
    return t


def loadPipeCrowdRun(name, collections, withTiming=True):
    butler = Butler(REPO, collections=list(collections) + list(INPUT_COLLECTIONS))
    exp = butler.get(VISIT_IMAGE, dataId=DATA_ID)
    model = butler.get(f"{OUTPUT_PREFIX}_model", dataId=DATA_ID)
    resid = butler.get(f"{OUTPUT_PREFIX}_residual", dataId=DATA_ID)
    src = butler.get(f"{OUTPUT_PREFIX}_src", dataId=DATA_ID)
    bbox, wcs = exp.getBBox(), exp.getWcs()
    calib = exp.getPhotoCalib()
    zp = calib.instFluxToMagnitude(1.0) if calib is not None else 31.4

    background = None
    for ds in BACKGROUND_DATASETS:
        try:
            background = butler.get(ds, dataId=DATA_ID).getImage().array.astype(np.float32)
            print(f"{name}: background from {ds}")
            break
        except Exception:
            continue
    if background is None:
        print(f"{name}: no background dataset found among {BACKGROUND_DATASETS}")

    timing = None
    if withTiming:
        try:
            timing = timingTable(butler.get(f"{TASK_LABEL}_metadata", dataId=DATA_ID))
            nStep = int(np.isin(timing["kind"], LEAF_KINDS).sum())
            if not len(timing):
                print(f"{name}: {TASK_LABEL}_metadata has no timing information")
                timing = None
            elif nStep == 0:
                print(f"{name}: only pipetask's whole-quantum timing; no step timings "
                      f"(the run was made with a pipe_crowd without the step timers)")
            else:
                print(f"{name}: {nStep} step timings")
        except Exception as e:
            print(f"{name}: no timing from {TASK_LABEL}_metadata ({type(e).__name__}: {e})")

    try:
        epoch = exp.visitInfo.date.toAstropy()
    except Exception:
        epoch = None
    run = dict(name=name, kind="pipe_crowd",
               data=exp.image.array.astype(np.float32),
               model=model.image.array.astype(np.float32),
               resid=resid.image.array.astype(np.float32),
               var=exp.variance.array.astype(np.float32),
               bad=badMask(exp.mask),
               x0=bbox.getMinX(), y0=bbox.getMinY(), shape=(bbox.getHeight(), bbox.getWidth()),
               cat=catalogTable(src, zp, wcs), zp=zp,
               pixScale=wcs.getPixelScale(bbox.getCenter()).asArcseconds(),
               psf=psfInfo(exp.getPsf(), bbox), background=background, timing=timing,
               epoch=epoch, wcs=wcs, bbox=bbox)
    print(f"{name}: {len(run['cat'])} sources, bbox {bbox}, ZP {zp:.3f}, "
          f"{run['pixScale']:.3f}\"/px, PSF FWHM {run['psf']['fwhmCenter']:.2f} px at the centre")
    return run


run = loadPipeCrowdRun(RUN_NAME, [RUN_COLLECTION])

### Gaia DR3 at the visit epoch

In [ ]:
def loadGaia(run):
    """Gaia stars on the detector from the Butler reference catalogue (proper motions applied).

    Returns a Table with ra, dec, G (AB, from the refcat flux), x, y.
    """
    if GAIA_CACHE and os.path.exists(GAIA_CACHE):
        return Table.read(GAIA_CACHE)
    import lsst.sphgeom as sph
    from lsst.meas.algorithms import ReferenceObjectLoader
    wcs, bbox = run["wcs"], run["bbox"]
    corners = [wcs.pixelToSky(geom.Point2D(p)) for p in bbox.getCorners()]
    poly = sph.ConvexPolygon.convexHull([sph.UnitVector3d(sph.LonLat.fromDegrees(
        c.getRa().asDegrees(), c.getDec().asDegrees())) for c in corners])
    shards = [i for b, e in sph.HtmPixelization(7).envelope(poly) for i in range(b, e)]
    butler = Butler(REPO)
    where = f"htm7 IN ({','.join(map(str, shards))})"
    try:
        refs = list(butler.query_datasets(GAIA_REFCAT, collections=GAIA_COLLECTIONS, where=where,
                                          with_dimension_records=True))
    except (TypeError, AttributeError):
        refs = list(butler.registry.queryDatasets(GAIA_REFCAT, collections=GAIA_COLLECTIONS,
                                                  where=where).expanded())
    loader = ReferenceObjectLoader(dataIds=[r.dataId for r in refs],
                                   refCats=[butler.getDeferred(r) for r in refs], name=GAIA_REFCAT)
    res = loader.loadPixelBox(bbox, wcs, filterName="phot_g_mean", epoch=run["epoch"])
    cat = res.refCat
    flux = np.asarray(cat[res.fluxField], float)
    with np.errstate(divide="ignore", invalid="ignore"):
        g = np.where(flux > 0, -2.5 * np.log10(flux) + 31.4, np.nan)
    t = Table(dict(ra=np.degrees(np.asarray(cat["coord_ra"], float)),
                   dec=np.degrees(np.asarray(cat["coord_dec"], float)), G=g))
    t["x"], t["y"] = wcs.skyToPixelArray(np.asarray(t["ra"]), np.asarray(t["dec"]), degrees=True)
    if GAIA_CACHE:
        os.makedirs(os.path.dirname(os.path.abspath(GAIA_CACHE)), exist_ok=True)
        t.write(GAIA_CACHE, overwrite=True)
    return t


try:
    gaia = loadGaia(run)
    print(f"Gaia: {len(gaia)} stars on the detector")
except Exception as e:
    gaia = None
    print(f"Gaia not loaded ({type(e).__name__}: {e}); the Gaia plots are skipped")

### Comparison runs

Every comparison run is a dict like `run`; entries it does not have are `None`.
`meas_crowd_v2` runs are read from the `stitchTiles` output directory (`detector_catalog.fits`, and
`detector_mosaic.fits` for the images if present).

In [ ]:
def loadMeasCrowd(name, outDir, ref):
    """meas_crowd_v2 detector run -> run dict (pixel frame = the visit_image's, like pipe_crowd)."""
    from astropy.io import fits
    t = Table.read(os.path.join(outDir, "detector_catalog.fits"))
    flux = np.asarray(t["psfFlux"], float)
    mag = np.asarray(t["psfMag"], float) if "psfMag" in t.colnames else \
        np.where(flux > 0, ref["zp"] - 2.5 * np.log10(np.clip(flux, 1e-30, None)), np.nan)
    cat = Table(dict(x=np.asarray(t["x"], float), y=np.asarray(t["y"], float), flux=flux, mag=mag))
    for c in ("ra", "dec"):
        if c in t.colnames:
            cat[c] = np.asarray(t[c], float)
    r = dict(name=name, kind="meas_crowd", cat=cat, zp=ref["zp"], pixScale=ref["pixScale"],
             x0=ref["x0"], y0=ref["y0"], shape=ref["shape"], psf=None, background=None,
             timing=None, data=None, model=None, resid=None, var=None, bad=None)
    mosaic = os.path.join(outDir, "detector_mosaic.fits")
    if os.path.exists(mosaic):
        with fits.open(mosaic) as hl:
            r["x0"], r["y0"] = int(hl["DATA"].header["X0"]), int(hl["DATA"].header["Y0"])
            r["data"], r["model"] = hl["DATA"].data.astype(np.float32), hl["MODEL"].data.astype(np.float32)
            r["resid"], r["var"] = hl["RESIDUAL"].data.astype(np.float32), hl["VARIANCE"].data.astype(np.float32)
            r["bad"] = (hl["BAD"].data != 0) | ~np.isfinite(r["model"])
            r["shape"] = r["data"].shape
    print(f"{name}: {len(cat)} sources" + ("" if r["data"] is None else ", with images"))
    return r


others = []
for name, coll in COMPARE_PIPE_CROWD.items():
    try:
        others.append(loadPipeCrowdRun(name, [coll]))
    except Exception as e:
        print(f"{name}: not loaded ({type(e).__name__}: {e})")
for name, d in MEAS_CROWD_DIRS.items():
    try:
        others.append(loadMeasCrowd(name, d, run))
    except Exception as e:
        print(f"{name}: not loaded ({type(e).__name__}: {e})")

In [ ]:
# Timing tables from files (e.g. meas_crowd_v2 StepTimer records), attached to the run of the same
# name or added as a timing-only run.
for name, path in EXTRA_TIMING.items():
    t = Table.read(path)
    if "kind" not in t.colnames:
        t["kind"] = "compute"
    for c, v in (("outer", -1), ("cpuSeconds", np.nan), ("startSeconds", -1.0), ("nSources", -1)):
        if c not in t.colnames:
            t[c] = v
    t["outer"] = [(-1 if (o is None or o is np.ma.masked) else int(o)) for o in t["outer"]]
    t = t[[not str(lbl).startswith("[nested]") for lbl in t["label"]]]
    match = [r for r in [run] + others if r["name"] == name]
    if match:
        match[0]["timing"] = t
    else:
        others.append(dict(name=name, kind="timing", cat=None, timing=t, data=None, psf=None,
                           background=None))

runs = [run] + others


def finitePositions(t, name):
    """Drop rows whose x or y is not finite (they break the matching and the kd-trees)."""
    ok = np.isfinite(np.asarray(t["x"], float)) & np.isfinite(np.asarray(t["y"], float))
    if not ok.all():
        print(f"{name}: dropped {np.sum(~ok)} of {len(t)} rows with non-finite x / y")
    return t[ok]


for r in runs:
    if r.get("cat") is not None:
        r["cat"] = finitePositions(r["cat"], r["name"])
if gaia is not None:
    gaia = finitePositions(gaia, "Gaia")
    (gxr, gyr) = detectorRange(run)
    onDet = (gaia["x"] >= gxr[0]) & (gaia["x"] <= gxr[1]) & (gaia["y"] >= gyr[0]) & (gaia["y"] <= gyr[1])
    if not onDet.all():
        print(f"Gaia: dropped {np.sum(~onDet)} stars off the detector")
    gaia = gaia[onDet]
withCat = [r for r in runs if r.get("cat") is not None]
if ZOOM is None:
    ZOOM = (run["x0"] + run["shape"][1] // 2 - 150, run["y0"] + run["shape"][0] // 2 - 150, 300)
print("runs:", [r["name"] for r in runs], "| zoom box:", ZOOM)

## 1. Timing

Rows of kind `setup`, `io` and `compute` add up; `wall` rows are totals that contain other rows
(`runQuantum (total)`, `run (total)`, `detection round (total)`, `centroid (total)`), and `quantum` rows
are pipetask's own measurements around the task.

In [ ]:
def leaf(t):
    return t[np.isin(t["kind"], LEAF_KINDS)]


def perStep(t):
    """Sum of seconds / CPU seconds / calls per label (leaf rows), in first-seen order."""
    t = leaf(t)
    labels = list(dict.fromkeys(t["label"]))
    out = Table(dict(label=labels,
                     kind=[t["kind"][t["label"] == lb][0] for lb in labels],
                     calls=[int(np.sum(t["label"] == lb)) for lb in labels],
                     seconds=[float(np.sum(t["seconds"][t["label"] == lb])) for lb in labels],
                     cpuSeconds=[float(np.nansum(t["cpuSeconds"][t["label"] == lb])) for lb in labels]))
    return out


def hasSteps(r):
    t = r.get("timing")
    return t is not None and bool(np.isin(t["kind"], LEAF_KINDS).any())


def runTime(r):
    """(seconds, what): the summed step time if the run has step timings, else pipetask's quantum total."""
    t = r.get("timing")
    if t is None or not len(t):
        return np.nan, ""
    if hasSteps(r):
        return float(np.sum(leaf(t)["seconds"])), "steps"
    q = t[t["label"] == "pipetask: quantum total"]
    return (float(q["seconds"][0]), "quantum") if len(q) else (np.nan, "")


def timingReport(r):
    t = r["timing"]
    q = t[t["kind"] == "quantum"]
    steps = perStep(t)
    total = float(np.sum(steps["seconds"]))
    print(f"=== {r['name']} ===")
    for row in q:
        print(f"  {row['label']:34s} {row['seconds']:9.2f} s wall  {row['cpuSeconds']:9.2f} s CPU")
    for lbl in ("runQuantum (total)", "run (total)"):
        sel = t["label"] == lbl
        if np.any(sel):
            print(f"  {lbl:34s} {float(np.sum(t['seconds'][sel])):9.2f} s wall")
    if len(steps):
        print(f"  {'sum of setup + io + compute':34s} {total:9.2f} s")
    else:
        print("  (no step timings in this run: only pipetask's whole-quantum timing above)")
        total, _ = runTime(r)
    if "peakRssMB" in t.colnames and len(t) and t["peakRssMB"][0] > 0:
        print(f"  peak memory {t['peakRssMB'][0] / 1e3:.2f} GB")
    if r.get("cat") is not None and total > 0:
        print(f"  {len(r['cat'])} sources -> {1e3 * total / max(len(r['cat']), 1):.2f} s per 1000 sources")


def plotTimingSteps(r):
    steps = perStep(r["timing"])
    steps.sort("seconds")
    total = float(np.sum(steps["seconds"]))
    fig, axs = plt.subplots(1, 2, figsize=(14, 0.32 * len(steps) + 1.5), sharey=True,
                            gridspec_kw=dict(width_ratios=[3, 1]))
    y = np.arange(len(steps))
    axs[0].barh(y, steps["seconds"], color=[KIND_COLORS.get(k, "k") for k in steps["kind"]])
    for yi, s, n in zip(y, steps["seconds"], steps["calls"]):
        axs[0].text(s, yi, f" {s:.2f} s ({100 * s / total:.0f}%)" + (f"  x{n}" if n > 1 else ""),
                    va="center", fontsize=8)
    axs[0].set_yticks(y, steps["label"], fontsize=8)
    axs[0].set_xlabel("wall time [s] (summed over rounds)")
    axs[0].set_title(f"{r['name']}: time per step (total {total:.1f} s)")
    axs[0].set_xlim(0, 1.3 * max(steps["seconds"].max(), 1e-3))
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = steps["cpuSeconds"] / steps["seconds"]
    axs[1].barh(y, ratio, color=BLUE_L)
    axs[1].axvline(1, color=INK, lw=0.8, ls="--")
    axs[1].set_xlabel("CPU / wall")
    axs[1].set_title("< 1: waiting (I/O); > 1: threads")
    axs[0].legend(handles=[Patch(color=KIND_COLORS[k], label=k) for k in LEAF_KINDS],
                  loc="lower right", fontsize=8)
    show(fig, f"timing_steps_{r['name']}")


def plotTimingRounds(r):
    t = leaf(r["timing"])
    t = t[t["outer"] >= 1]
    if not len(t):
        return
    rounds = sorted(set(int(o) for o in t["outer"]))
    labels = list(dict.fromkeys(t["label"]))
    stepColors = vcolors(len(labels))
    fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))
    bottom = np.zeros(len(rounds))
    for i, lb in enumerate(labels):
        v = np.array([np.sum(t["seconds"][(t["label"] == lb) & (t["outer"] == o)]) for o in rounds])
        axs[0].bar(rounds, v, bottom=bottom, color=stepColors[i], label=lb)
        bottom += v
    axs[0].set_xticks(rounds)
    axs[0].set_xlabel("detection round")
    axs[0].set_ylabel("wall time [s]")
    axs[0].set_title(f"{r['name']}: time per detection round")
    axs[0].legend(fontsize=7, bbox_to_anchor=(1.0, 1.0), loc="upper left")
    # scaling with the number of sources: each step's time vs the sources it worked on
    t2 = t[t["nSources"] > 0]
    for i, lb in enumerate(labels):
        s = t2[t2["label"] == lb]
        if len(s):
            axs[1].plot(s["nSources"], s["seconds"], "o-", color=stepColors[i], label=lb, ms=4)
    axs[1].set_xlabel("sources in the step")
    axs[1].set_ylabel("wall time [s]")
    axs[1].set_title("scaling with source count (one point per round)")
    if len(t2):
        axs[1].set_xscale("log")
        axs[1].set_yscale("log")
    show(fig, f"timing_rounds_{r['name']}")


def plotTimeline(r):
    t = r["timing"]
    t = t[(t["startSeconds"] >= 0) & (t["kind"] != "quantum")]
    if not len(t):
        return
    labels = list(dict.fromkeys(t["label"]))
    fig, ax = plt.subplots(figsize=(14, 0.28 * len(labels) + 1.5))
    for row in t:
        yi = labels.index(row["label"])
        wall = row["kind"] == "wall"
        ax.barh(yi, row["seconds"], left=row["startSeconds"], height=0.7,
                color="none" if wall else KIND_COLORS.get(row["kind"], "k"),
                edgecolor=KIND_COLORS["wall"] if wall else "none", lw=1.2)
        if row["outer"] >= 1 and not wall and row["seconds"] > 0.02 * t["seconds"].max():
            ax.text(row["startSeconds"] + row["seconds"] / 2, yi, str(row["outer"]),
                    ha="center", va="center", fontsize=7)
    ax.set_yticks(range(len(labels)), labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("seconds since task construction")
    ax.set_title(f"{r['name']}: timeline (outlined = totals; numbers = detection round)")
    show(fig, f"timing_timeline_{r['name']}")


for r in runs:
    if r.get("timing") is not None and len(r["timing"]):
        timingReport(r)
        if hasSteps(r):
            plotTimingSteps(r)
            if r["kind"] == "pipe_crowd":
                plotTimingRounds(r)
            plotTimeline(r)

### Timing comparison between runs

Steps are grouped into broad phases (by keywords in their labels: edit `PHASES` to change this)
so runs of different algorithms, e.g. `pipe_crowd` and `meas_crowd_v2`, can be put side by side.

In [ ]:
PHASES = [   # (phase, regular expression on the lower-case label); the first match wins
    ("read / write", r"read|write|save|load|butler"),
    ("setup", r"setup|construct"),
    ("model / subtract", r"convolve"),
    ("centroid", r"centroid|polish|position"),
    ("PSF", r"psf|piff"),
    ("background", r"background|sky"),
    ("detection", r"detect|peak"),
    ("fit", r"matrix|joint|lsq|solve|fit"),
    ("model / subtract", r"model|subtract|residual"),
    ("cleanup / merge", r"drop|merge|reject|collapse|nan|pair"),
]
PHASE_NAMES = list(dict.fromkeys([p for p, _ in PHASES] + ["other"]))
PHASE_COLORS = dict(zip(PHASE_NAMES, vcolors(len(PHASE_NAMES))))


def phaseOf(label):
    lb = str(label).lower()
    for p, rx in PHASES:
        if re.search(rx, lb):
            return p
    return "other"


timed = [r for r in runs if hasSteps(r)]
if len(timed) > 1:
    fig, axs = plt.subplots(1, 2, figsize=(14, 0.6 * len(timed) + 2.5))
    names = [r["name"] for r in timed]
    left = np.zeros(len(timed))
    leftK = np.zeros(len(timed))
    perPhase = {}
    for p in PHASE_NAMES:
        v = np.array([sum(float(s) for lb, s in zip(leaf(r["timing"])["label"], leaf(r["timing"])["seconds"])
                          if phaseOf(lb) == p) for r in timed])
        perPhase[p] = v
        axs[0].barh(names, v, left=left, color=PHASE_COLORS[p], label=p)
        left += v
        nSrc = np.array([len(r["cat"]) if r.get("cat") is not None else np.nan for r in timed])
        vk = 1e3 * v / nSrc
        axs[1].barh(names, np.nan_to_num(vk), left=leftK, color=PHASE_COLORS[p])
        leftK += np.nan_to_num(vk)
    axs[0].set_xlabel("wall time [s]")
    axs[0].set_title("time per phase")
    fig.legend(handles=[Patch(color=PHASE_COLORS[p], label=p) for p in PHASE_NAMES],
               loc="outside lower center", ncols=len(PHASE_NAMES))
    axs[1].set_xlabel("seconds per 1000 sources")
    axs[1].set_title("normalised by the number of sources found")
    axs[1].set_yticks([])
    show(fig, "timing_comparison")
    print(Table({"phase": PHASE_NAMES, **{n: np.round([perPhase[p][i] for p in PHASE_NAMES], 2)
                                           for i, n in enumerate(names)}}))
else:
    print("Fewer than two runs with step timings: nothing to compare per phase "
          "(add COMPARE_PIPE_CROWD / EXTRA_TIMING entries, or rerun with the step timers).")

## 2. Data / model / residual

Whole detector (block-averaged), then the zoom box at full resolution, each twice: once with the
residual and χ in grayscale like the data, once with them in a diverging colour map with white at 0
(red = data above the model, blue = below). χ = residual / √variance; in the block-averaged view it
is multiplied by the block size so that pure noise still has σ ≈ 1. Masked pixels are light blue in the
grayscale panels and grey in the diverging ones.

In [ ]:
def residualStyle(img, style, chi=False):
    """imshow keywords for a residual (or χ) panel: 'gray' or 'diverging' (symmetric, white = 0)."""
    lim = dict(vmin=-5, vmax=5) if chi else symNorm(img)
    return dict(cmap=GRAY if style == "gray" else DIVERGING, **lim)


def plotDMR(r, zoom=None, style="gray"):
    if r.get("data") is None:
        print(f"{r['name']}: no images")
        return
    chi = chiImage(r)
    imgs = [masked(r, "data"), masked(r, "model"), masked(r, "resid"), chi]
    if zoom is None:
        views = [fullView(r, im) for im in imgs]
        f = views[0][2]
        views[3] = (views[3][0] * f, views[3][1], f)     # binned chi rescaled to unit variance
        title = f"{r['name']}: whole detector (binned {f}×{f})"
    else:
        views = [cut(r, im, zoom) + (1,) for im in imgs]
        if views[0][0] is None:
            print(f"{r['name']}: zoom box outside the images")
            return
        title = f"{r['name']}: zoom {zoom}"
    title += " — residual " + ("grayscale" if style == "gray" else "diverging (white = 0)")
    norm = imageNorm(views[0][0])
    fig, axs = plt.subplots(1, 4, figsize=(20, 5.2), sharex=True, sharey=True)
    panels = [("data", dict(norm=norm, cmap=GRAY), FLUX_LABEL),
              ("model", dict(norm=norm, cmap=GRAY), FLUX_LABEL),
              ("residual (data − model)", residualStyle(views[2][0], style), FLUX_LABEL),
              ("χ = residual / σ", residualStyle(views[3][0], style, chi=True), "χ")]
    for ax, (im, ext, _), (lab, kw, cbl) in zip(axs, views, panels):
        h = ax.imshow(im, extent=ext, **kw)
        ax.set_title(lab)
        plt.colorbar(h, ax=ax, fraction=0.046, pad=0.02, label=cbl)
    if zoom is None:
        zx, zy, s = ZOOM
        axs[0].add_patch(Rectangle((zx, zy), s, s, fill=False, ec=RED, lw=1.8, ls="--"))
    fig.suptitle(title)
    show(fig, f"dmr_{'full' if zoom is None else 'zoom'}_{style}_{r['name']}")


def plotChiHist(rs):
    fig, axs = plt.subplots(1, 2, figsize=(13, 4))
    bins = np.linspace(-10, 10, 201)
    g = np.exp(-0.5 * (0.5 * (bins[1:] + bins[:-1])) ** 2) / np.sqrt(2 * np.pi)
    rows = []
    for i, r in enumerate(rs):
        chi = chiImage(r)
        if chi is None:
            continue
        sig = r["model"] > 3 * np.sqrt(r["var"])     # pixels where the model is significant
        for ax, sel, what in ((axs[0], np.isfinite(chi), "all good pixels"),
                              (axs[1], np.isfinite(chi) & sig, "pixels with model > 3σ")):
            v = chi[sel]
            if v.size:
                ax.hist(v, bins=bins, density=True, histtype="step", color=RUN_COLORS[i % 6], label=r["name"])
                ax.set_title(what)
        v = chi[np.isfinite(chi)]
        vs = chi[np.isfinite(chi) & sig]
        mad = lambda a: 1.4826 * np.median(np.abs(a - np.median(a))) if a.size else np.nan
        rows.append((r["name"], np.median(v), mad(v), np.mean(np.abs(v) > 5), np.median(vs) if vs.size else np.nan,
                     mad(vs), np.mean(np.abs(vs) > 5) if vs.size else np.nan))
    for ax in axs:
        ax.plot(0.5 * (bins[1:] + bins[:-1]), g, ls="--", color=INK, lw=0.9, label="N(0, 1)")
        ax.set_yscale("log")
        ax.set_ylim(1e-6, 1.0)
        ax.set_xlabel("χ")
        ax.legend(fontsize=8)
    show(fig, "chi_hist")
    t = Table(rows=rows, names=["run", "median χ", "σ_MAD χ", "frac |χ|>5", "median χ (stars)",
                                "σ_MAD χ (stars)", "frac |χ|>5 (stars)"])
    for c in t.colnames[1:]:
        t[c].format = ".3f"
    print(t)


for r in runs:
    for style in ("gray", "diverging"):
        plotDMR(r, None, style)
for r in runs:
    for style in ("gray", "diverging"):
        plotDMR(r, ZOOM, style)
plotChiHist([r for r in runs if r.get("resid") is not None])

## 3. Background

`pipe_crowd` fits sources on a `visit_image` whose sky background was already subtracted by the
upstream pipeline, and it fits no sky of its own. So any error in that background goes straight into
the fitted fluxes. These three panels check it:

* **Left: the subtracted background.** The smooth sky model the upstream pipeline removed
  (`visit_image_background`). It should be smooth; structure that follows the star field means
  starlight was absorbed into the "background".
* **Middle: the sky left in the residual.** In each `SKY_BIN_PX` bin (red > 0, blue < 0, white = 0), the median of the residual
  (data − model) over "sky" pixels, i.e. pixels where the model is below 1σ of the noise. If the
  background were right and every source were fitted, this would be ≈ 0 everywhere. Positive =
  light left over (sky under-subtracted, or faint sources nobody fitted); negative =
  over-subtracted sky, which in crowded fields typically happens because the background estimate
  is pulled up by the unresolved stars.
* **Right: residual sky vs crowding.** Each dot is one bin of the middle map, plotted against the
  number of sources per arcmin² in that bin; the line is the running median (band: 16–84 %). A flat
  line at 0 is good. A line going down with density means the background was over-subtracted where
  the field is crowded, which biases faint stars there low (and makes them harder to detect).

In [ ]:
def skyMap(r):
    """Binned median of the residual where the model is faint, and the matching source density."""
    with np.errstate(invalid="ignore"):
        faint = r["model"] < np.sqrt(r["var"])
    sky = masked(r, "resid")
    sky[~faint] = np.nan
    b, used = block(sky, SKY_BIN_PX, np.nanmedian)
    ext = extent(r["x0"], r["y0"], used)
    dens = None
    if r.get("cat") is not None:
        dens, _, _ = np.histogram2d(r["cat"]["y"], r["cat"]["x"], bins=b.shape,
                                    range=[ext[2:], ext[:2]])
        dens = dens / areaArcmin2(r, SKY_BIN_PX ** 2)
    return b, ext, dens


def plotBackground(r):
    if r.get("resid") is None:
        return
    fig, axs = plt.subplots(1, 3, figsize=(19, 5.5))
    if r.get("background") is not None:
        bg, ext, f = fullView(r, r["background"])
        h = axs[0].imshow(bg, extent=ext, cmap=SEQMAP)
        plt.colorbar(h, ax=axs[0], fraction=0.046, pad=0.02, label="background [image units]")
        axs[0].set_title(f"subtracted background (mean {np.nanmean(r['background']):.1f}, "
                         f"range {np.nanmin(r['background']):.1f} .. {np.nanmax(r['background']):.1f})")
    else:
        axs[0].text(0.5, 0.5, "no background dataset", ha="center", transform=axs[0].transAxes)
        axs[0].set_axis_off()
    sky, ext, dens = skyMap(r)
    h = axs[1].imshow(sky, extent=ext, cmap=DIVERGING, **symNorm(sky, 98))
    plt.colorbar(h, ax=axs[1], fraction=0.046, pad=0.02, label=FLUX_LABEL)
    axs[1].set_title(f"residual sky ({SKY_BIN_PX} px bins, median {np.nanmedian(sky):.2f})")
    if dens is not None:
        ok = np.isfinite(sky)
        axs[2].plot(dens[ok], sky[ok], ".", ms=3, alpha=0.5, color=BLUE_L)
        c, m, lo, hi = runningMedian(dens[ok], sky[ok], np.linspace(0, np.nanmax(dens) + 1e-9, 15))
        axs[2].plot(c, m, "-", color=RED, label="running median")
        axs[2].fill_between(c, lo, hi, color=RED, alpha=0.15, lw=0)
        axs[2].axhline(0, color=INK, lw=0.8)
        axs[2].set_xlabel("sources / arcmin²")
        axs[2].set_ylabel(f"residual sky: {FLUX_LABEL}")
        axs[2].legend(fontsize=8)
        axs[2].set_title("does the residual sky depend on crowding?")
    fig.suptitle(r["name"])
    show(fig, f"background_{r['name']}")


for r in runs:
    plotBackground(r)

## 4. Source catalogue over data / model / residual (zoom box)

Same colour scheme as section 2 (grayscale, and a second version with a diverging residual).
Red circles: final centroids. Blue `+`: the coarse (detection-peak) centroids of `pipe_crowd`, so the
centroid shifts are visible too.

In [ ]:
def plotOverlay(r, zoom, style="gray"):
    imgs = [cut(r, masked(r, k), zoom) for k in ("data", "model", "resid")]
    if imgs[0][0] is None:
        print(f"{r['name']}: no images in the zoom box")
        return
    cat = r["cat"][inZoom(r["cat"], zoom)]
    norm = imageNorm(imgs[0][0])
    fig, axs = plt.subplots(1, 3, figsize=(19, 6.5), sharex=True, sharey=True)
    for ax, (im, ext), lab, kw in zip(axs, imgs, ["data", "model", "residual (data − model)"],
                                      [dict(norm=norm, cmap=GRAY), dict(norm=norm, cmap=GRAY),
                                       residualStyle(imgs[2][0], style)]):
        h = ax.imshow(im, extent=ext, **kw)
        plt.colorbar(h, ax=ax, fraction=0.046, pad=0.02, label=FLUX_LABEL)
        ax.plot(cat["x"], cat["y"], **MK_SRC, label="final centroid")
        if "xCoarse" in cat.colnames:
            ax.plot(cat["xCoarse"], cat["yCoarse"], **MK_PEAK, label="detection peak")
        ax.set_title(f"{lab}: {len(cat)} sources")
        ax.set_xlim(ext[0], ext[1])
        ax.set_ylim(ext[2], ext[3])
    axs[0].legend(**LEGEND_ON_IMAGE)
    fig.suptitle(f"{r['name']}: zoom {zoom} — residual "
                 + ("grayscale" if style == "gray" else "diverging (white = 0)"))
    show(fig, f"overlay_{style}_{r['name']}")


for r in withCat:
    for style in ("gray", "diverging"):
        plotOverlay(r, ZOOM, style)

## 5. PSF model

The visit's PSF model (the one `pipe_crowd` fits with): kernel and radial profile at the detector
centre, how it changes across the detector (kernel minus the central kernel on a 3×3 grid), the FWHM
map, and stacks of bright isolated stars in the data and the residual. A ring or a positive / negative
core in the stacked residual means the PSF model does not match the stars. In the x / y cross-sections:
a positive peak at 0 with negative wings = stars more peaked (narrower) than the model; a negative
centre with positive wings = stars broader than the model; an S shape across 0 = centroid offset.

In [ ]:
def radialProfile(k):
    ny, nx = k.shape
    yy, xx = np.indices(k.shape)
    cy, cx = np.unravel_index(np.argmax(k), k.shape)
    rr = np.hypot(xx - cx, yy - cy).ravel()
    order = np.argsort(rr)
    return rr[order], k.ravel()[order] / k.max()


def starStack(r, half=15, nMax=200, isolation=10.0):
    """Median of flux-normalised data and residual stamps of bright isolated sources."""
    cat = r["cat"]
    if r.get("data") is None or len(cat) < 2:
        return None
    xy = np.c_[cat["x"], cat["y"]]
    d, _ = cKDTree(xy).query(xy, k=2)
    ix = np.round(cat["x"] - r["x0"]).astype(int)
    iy = np.round(cat["y"] - r["y0"]).astype(int)
    ny, nx = r["data"].shape
    ok = ((d[:, 1] > isolation) & (cat["flux"] > 0) & (ix >= half) & (iy >= half) &
          (ix < nx - half) & (iy < ny - half))
    idx = np.flatnonzero(ok)
    # brightest, skipping the very top few percent (likely saturated)
    idx = idx[np.argsort(-cat["flux"][idx])]
    idx = idx[int(0.05 * len(idx)):][:nMax]
    data, resid = [], []
    for i in idx:
        sl = (slice(iy[i] - half, iy[i] + half + 1), slice(ix[i] - half, ix[i] + half + 1))
        if r["bad"][sl].any():
            continue
        data.append(r["data"][sl] / cat["flux"][i])
        resid.append(r["resid"][sl] / cat["flux"][i])
    if len(data) < 5:
        return None
    return dict(n=len(data), data=np.median(data, axis=0), resid=np.median(resid, axis=0))


def plotPsf(r):
    p = r["psf"]
    fig, axs = plt.subplots(1, 3, figsize=(17, 5))
    k = p["center"]
    h = axs[0].imshow(k / k.max(), norm=LogNorm(1e-4, 1), cmap=GRAYLOG)
    plt.colorbar(h, ax=axs[0], fraction=0.046, pad=0.02, label="normalised to peak (log)")
    axs[0].set_title(f"PSF kernel at ({p['centerXY'][0]:.0f}, {p['centerXY'][1]:.0f}), "
                     f"FWHM {p['fwhmCenter']:.2f} px = {p['fwhmCenter'] * r['pixScale']:.2f}\"")
    rr, v = radialProfile(k)
    axs[1].semilogy(rr, v, "o", ms=3.5, color=BLUE, label="PSF model", zorder=3)
    sig = p["fwhmCenter"] / 2.3548
    rg = np.linspace(0, rr.max(), 200)
    axs[1].semilogy(rg, np.exp(-0.5 * (rg / sig) ** 2), ls="--", color="#8a8a8a", lw=1.2,
                    label="Gaussian, same FWHM", zorder=2)
    st = starStack(r)
    if st is not None:
        rs, vs = radialProfile(st["data"])
        axs[1].semilogy(rs, vs, "s", ms=3.5, alpha=0.7, color=RED, mew=0,
                        label=f"stacked stars (n={st['n']})", zorder=1)
    axs[1].set_ylim(1e-5, 1.5)
    axs[1].set_xlabel("radius [px]")
    axs[1].set_ylabel("normalised")
    axs[1].legend(fontsize=8)
    axs[1].set_title("radial profile")
    h = axs[2].imshow(p["fwhmMap"] * r["pixScale"], cmap=SEQMAP, aspect="auto",
                      extent=(p["fwhmX"][0], p["fwhmX"][-1], p["fwhmY"][0], p["fwhmY"][-1]))
    plt.colorbar(h, ax=axs[2], fraction=0.046, pad=0.02, label="FWHM [arcsec]")
    axs[2].set_title(f"FWHM map ({p['fwhmMap'].min() * r['pixScale']:.3f} .. "
                     f"{p['fwhmMap'].max() * r['pixScale']:.3f}\")")
    fig.suptitle(r["name"])
    show(fig, f"psf_{r['name']}")

    # spatial variation: grid kernels minus the central kernel
    grid = p["grid"]
    n = len(grid)
    fig, axs = plt.subplots(n, n, figsize=(2.6 * n, 2.6 * n))
    for j in range(n):
        for i in range(n):
            g = grid[j][i]
            s = min(g.shape[0], k.shape[0]) // 2
            gc, kc = g.shape[0] // 2, k.shape[0] // 2
            diff = g[gc - s:gc + s + 1, gc - s:gc + s + 1] / g.sum() - k[kc - s:kc + s + 1, kc - s:kc + s + 1] / k.sum()
            ax = axs[n - 1 - j, i]
            lim = 0.02 * k.max() / k.sum()
            hh = ax.imshow(diff / (k.max() / k.sum()), cmap=DIVERGING, vmin=-0.02, vmax=0.02)
            ax.set_title(f"({p['gridX'][i]:.0f}, {p['gridY'][j]:.0f})", fontsize=8)
            ax.set_xticks([])
            ax.set_yticks([])
    fig.colorbar(hh, ax=axs, fraction=0.04, label="difference / central peak")
    fig.suptitle(f"{r['name']}: PSF kernel minus central kernel (±2% of peak)")
    show(fig, f"psf_grid_{r['name']}")


def plotStarStacks(rs):
    stacks = [(r, starStack(r)) for r in rs]
    stacks = [(r, s) for r, s in stacks if s is not None]
    if not stacks:
        return
    fig, axs = plt.subplots(len(stacks), 3, figsize=(13, 4 * len(stacks)), squeeze=False)
    for (r, s), row in zip(stacks, axs):
        pk = s["data"].max()
        h = row[0].imshow(s["data"] / pk, norm=LogNorm(1e-4, 1), cmap=GRAYLOG)
        plt.colorbar(h, ax=row[0], fraction=0.046, label="normalised to peak (log)")
        row[0].set_title(f"{r['name']}: stacked stars (n={s['n']})")
        h = row[1].imshow(s["resid"] / pk, cmap=DIVERGING, vmin=-0.05, vmax=0.05)
        plt.colorbar(h, ax=row[1], fraction=0.046)
        row[1].set_title("stacked residual / star peak (±5%)")
        c = s["resid"].shape[0] // 2
        px = np.arange(-c, c + 1)
        row[2].plot(px, s["resid"][c] / pk, "-o", ms=3.5, color=BLUE, label="x direction (row through centre)")
        row[2].plot(px, s["resid"][:, c] / pk, "--s", ms=3.5, color=RED,
                    label="y direction (column through centre)")
        row[2].axhline(0, color=INK, lw=0.8)
        row[2].set_xlabel("offset from star centre [px]")
        row[2].set_ylabel("residual / star peak")
        row[2].set_title("PSF cross-sections in x and y\nstacked residual (data − model) through the star centre")
        row[2].legend(fontsize=8)
    show(fig, "star_stacks")


for r in runs:
    if r.get("psf") is not None:
        plotPsf(r)
plotStarStacks([r for r in withCat if r.get("data") is not None])

## 6. Gaia

Gaia DR3 positions (at the visit epoch) over the zoom box, then for every run: completeness against
Gaia vs G, astrometric offsets (scatter and a map of binned median offsets: a pattern means a WCS
or centroid bias), and fitted magnitude minus Gaia G. Gaia G and the LSST band differ, so only the
spread and trends of `mag − G` matter, not its offset.

In [ ]:
def gaiaMatch(r):
    cat = r["cat"]
    ia, ig, d = matchXY(cat["x"], cat["y"], gaia["x"], gaia["y"], MATCH_RADIUS_PX)
    return ia, ig, d


def plotGaiaOverlay(r, zoom):
    im, ext = cut(r, masked(r, "data"), zoom)
    if im is None:
        return
    cat = r["cat"]
    ia, ig, _ = gaiaMatch(r)
    gz = inZoom(gaia, zoom)
    matchedG = np.zeros(len(gaia), bool)
    matchedG[ig] = True
    fig, ax = plt.subplots(figsize=(9, 9))
    h = ax.imshow(im, extent=ext, norm=imageNorm(im), cmap=GRAY)
    plt.colorbar(h, ax=ax, fraction=0.046, pad=0.02, label=FLUX_LABEL)
    cz = inZoom(cat, zoom)
    ax.plot(cat["x"][cz], cat["y"][cz], **MK_SRC, label=f"{r['name']} source ({cz.sum()})")
    ax.plot(gaia["x"][gz & matchedG], gaia["y"][gz & matchedG], **MK_GAIA,
            label=f"Gaia, matched ({(gz & matchedG).sum()})")
    ax.plot(gaia["x"][gz & ~matchedG], gaia["y"][gz & ~matchedG], **MK_GAIA_MISSED,
            label=f"Gaia, NOT matched ({(gz & ~matchedG).sum()})")
    for x, y, g in zip(gaia["x"][gz & ~matchedG], gaia["y"][gz & ~matchedG], gaia["G"][gz & ~matchedG]):
        ax.annotate(f"G={g:.1f}", (x, y), xytext=(6, 6), textcoords="offset points", fontsize=8,
                    color="white", path_effects=[pe.withStroke(linewidth=2.4, foreground=BLUE)])
    ax.set_xlim(ext[0], ext[1])
    ax.set_ylim(ext[2], ext[3])
    ax.legend(**LEGEND_ON_IMAGE)
    ax.set_title(f"{r['name']} vs Gaia within {MATCH_RADIUS_PX} px (labels: G of unmatched Gaia stars)")
    show(fig, f"gaia_overlay_{r['name']}")


def plotGaiaStats(rs):
    gbins = np.arange(np.floor(np.nanmin(gaia["G"])), np.ceil(np.nanmax(gaia["G"])) + 0.5, 0.5)
    gc = 0.5 * (gbins[1:] + gbins[:-1])
    nG, _ = np.histogram(gaia["G"], gbins)
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    rows = []
    for i, r in enumerate(rs):
        col = RUN_COLORS[i % 6]
        ia, ig, d = gaiaMatch(r)
        nm, _ = np.histogram(gaia["G"][ig], gbins)
        with np.errstate(invalid="ignore", divide="ignore"):
            axs[0, 0].plot(gc, nm / nG, "o-", color=col, ms=3, label=r["name"])
        dx = r["cat"]["x"][ia] - gaia["x"][ig]
        dy = r["cat"]["y"][ia] - gaia["y"][ig]
        bright = gaia["G"][ig] < 19
        axs[0, 1].plot(dx[bright], dy[bright], "o", color=col, ms=3, alpha=0.5, mew=0,
                       label=f"{r['name']}: median ({np.median(dx[bright]) if bright.any() else np.nan:+.3f}, "
                             f"{np.median(dy[bright]) if bright.any() else np.nan:+.3f}) px")
        dm = r["cat"]["mag"][ia] - gaia["G"][ig]
        off = np.nanmedian(dm[bright]) if bright.any() else np.nanmedian(dm)
        axs[1, 0].plot(gaia["G"][ig], dm - off, ".", color=col, ms=2, alpha=0.3)
        c, m, lo, hi = runningMedian(np.asarray(gaia["G"][ig]), np.asarray(dm - off), gbins)
        axs[1, 0].plot(c, m, "-", color=col, lw=2, label=f"{r['name']} (offset {off:+.2f} removed)")
        axs[1, 1].hist(d * r["pixScale"] * 1e3, bins=50, range=(0, MATCH_RADIUS_PX * r["pixScale"] * 1e3),
                       histtype="step", color=col, label=r["name"])
        comp = lambda gmax: (np.sum(gaia["G"][ig] < gmax) / max(np.sum(gaia["G"] < gmax), 1))
        mad = lambda a: 1.4826 * np.median(np.abs(a - np.median(a))) if len(a) else np.nan
        rows.append((r["name"], len(r["cat"]), len(ia), comp(18), comp(19), comp(20), comp(21),
                     mad(dx[bright]) * r["pixScale"] * 1e3, mad(dy[bright]) * r["pixScale"] * 1e3,
                     mad(np.asarray(dm[bright])) if bright.any() else np.nan))
    axs[0, 0].set_xlabel("Gaia G")
    axs[0, 0].set_ylabel(f"fraction of Gaia stars matched (< {MATCH_RADIUS_PX} px)")
    axs[0, 0].set_ylim(0, 1.05)
    axs[0, 0].grid(alpha=0.3)
    axs[0, 0].set_title("completeness vs Gaia")
    ax2 = axs[0, 0].twinx()
    ax2.bar(gc, nG, width=0.5, color=BLUE_L, alpha=0.3, zorder=0)
    ax2.set_ylabel("Gaia stars per bin", color="#777777")
    ax2.spines["right"].set_visible(True)
    ax2.grid(False)
    axs[0, 0].set_zorder(ax2.get_zorder() + 1)
    axs[0, 0].patch.set_visible(False)
    axs[0, 0].legend(fontsize=8)
    lim = MATCH_RADIUS_PX
    axs[0, 1].set_xlim(-lim, lim)
    axs[0, 1].set_ylim(-lim, lim)
    axs[0, 1].set_aspect("equal")
    axs[0, 1].axhline(0, color=INK, lw=0.5)
    axs[0, 1].axvline(0, color=INK, lw=0.5)
    axs[0, 1].set_xlabel("x − x_Gaia [px]")
    axs[0, 1].set_ylabel("y − y_Gaia [px]")
    axs[0, 1].set_title("astrometric offsets (G < 19)")
    axs[0, 1].legend(fontsize=7)
    axs[1, 0].axhline(0, color=INK, lw=0.8)
    axs[1, 0].set_ylim(-1, 1)
    axs[1, 0].set_xlabel("Gaia G")
    axs[1, 0].set_ylabel("mag − G − median offset")
    axs[1, 0].set_title("photometry vs Gaia (line: running median)")
    axs[1, 0].legend(fontsize=8)
    axs[1, 1].set_xlabel("separation [mas]")
    axs[1, 1].set_title("match separations")
    axs[1, 1].legend(fontsize=8)
    show(fig, "gaia_stats")
    t = Table(rows=rows, names=["run", "N sources", "N Gaia matched", "compl. G<18", "compl. G<19",
                                "compl. G<20", "compl. G<21", "σ_x [mas]", "σ_y [mas]", "σ(mag−G)"])
    for c in t.colnames[3:]:
        t[c].format = ".3f"
    print(t)
    return t


def plotOffsetMap(r, nBin=8):
    ia, ig, _ = gaiaMatch(r)
    sel = gaia["G"][ig] < 20
    ia, ig = ia[sel], ig[sel]
    if len(ia) < 20:
        return
    dx = r["cat"]["x"][ia] - gaia["x"][ig]
    dy = r["cat"]["y"][ia] - gaia["y"][ig]
    (xr, yr) = detectorRange(r)
    xe, ye = np.linspace(*xr, nBin + 1), np.linspace(*yr, nBin + 1)
    bx = np.clip(np.digitize(gaia["x"][ig], xe) - 1, 0, nBin - 1)
    by = np.clip(np.digitize(gaia["y"][ig], ye) - 1, 0, nBin - 1)
    mx, my, n = (np.full((nBin, nBin), np.nan) for _ in range(3))
    for j in range(nBin):
        for i in range(nBin):
            s = (bx == i) & (by == j)
            n[j, i] = s.sum()
            if s.sum() >= 5:
                mx[j, i], my[j, i] = np.median(dx[s]), np.median(dy[s])
    xc, yc = 0.5 * (xe[1:] + xe[:-1]), 0.5 * (ye[1:] + ye[:-1])
    X, Y = np.meshgrid(xc, yc)
    fig, ax = plt.subplots(figsize=(8, 8))
    amp = np.hypot(mx, my) * r["pixScale"] * 1e3
    h = ax.imshow(amp, extent=(xr[0], xr[1], yr[0], yr[1]), cmap=SEQMAP, alpha=0.6)
    plt.colorbar(h, ax=ax, fraction=0.046, label="|median offset| [mas]")
    scale = (xr[1] - xr[0]) / nBin / max(np.nanmax(np.hypot(mx, my)), 1e-3) * 0.8
    ax.quiver(X, Y, mx * scale, my * scale, angles="xy", scale_units="xy", scale=1, color="white",
              edgecolor=INK, linewidth=0.6)
    ax.set_title(f"{r['name']}: median offset from Gaia per bin (G < 20), arrows ×{scale:.0f}")
    show(fig, f"gaia_offset_map_{r['name']}")


gaiaSummary = None
if gaia is not None and len(gaia):
    for r in withCat:
        if r.get("data") is not None:
            plotGaiaOverlay(r, ZOOM)
    gaiaSummary = plotGaiaStats(withCat)
    for r in withCat:
        plotOffsetMap(r)
else:
    print("no Gaia catalogue")

## 7. Source density over the whole detector

The detector is cut into square bins of `DENSITY_BIN_PX` pixels (128 px ≈ 26″ on LSSTCam), and every
panel shows one number per bin:

* **Top row: how many sources each run found per arcmin².** Counts are divided by the *unmasked* area
  of the bin, so bad columns and bleed trails don't fake a low density. All runs and Gaia share
  one colour scale. A real crowded field shows smooth large-scale structure (the stellar
  density gradient); sharp edges or stripes point to detector / processing artefacts. Gaia is much
  shallower, so its map is dimmer: it shows the bright-star distribution only.
* **Masked fraction** (second row, first panel): the fraction of each bin's pixels that are masked.
  Bins less than 25 % usable are left blank (grey) in the other maps.
* **Run − main run** (diverging, white = 0): where a comparison run finds more (red) or fewer (blue)
  sources per arcmin² than the main run. A uniform tint is a depth difference; a pattern that
  follows the density map means the two runs handle crowding differently.
* **Gaia completeness map** (per run): the fraction of Gaia stars in each bin that the run matched
  within `MATCH_RADIUS_PX` (blank where a bin has fewer than 3 Gaia stars). Gaia stars are bright,
  so this should be ≈ 1 everywhere; lower values show where the run loses even bright stars
  (typically the most crowded bins, or near masked regions).

In [ ]:
def densityGrid():
    """Bins of DENSITY_BIN_PX over the main run's detector: fraction of each bin that is on the
    detector and unmasked, fraction on the detector, and the bin edges' extent."""
    B = DENSITY_BIN_PX
    ny, nx = run["shape"]
    nby, nbx = int(np.ceil(ny / B)), int(np.ceil(nx / B))
    good = np.zeros((nby * B, nbx * B))
    cov = np.zeros_like(good)
    good[:ny, :nx] = ~run["bad"] if run.get("bad") is not None else 1.0
    cov[:ny, :nx] = 1.0
    goodFrac = good.reshape(nby, B, nbx, B).mean(axis=(1, 3))
    covFrac = cov.reshape(nby, B, nbx, B).mean(axis=(1, 3))
    ext = (run["x0"] - 0.5, run["x0"] - 0.5 + nbx * B, run["y0"] - 0.5, run["y0"] - 0.5 + nby * B)
    return goodFrac, covFrac, ext


GOOD_FRAC, COV_FRAC, DENS_EXT = densityGrid()


def binCounts(x, y):
    h, _, _ = np.histogram2d(y, x, bins=GOOD_FRAC.shape, range=[DENS_EXT[2:], DENS_EXT[:2]])
    return h


def densityMap(cat):
    """Sources per unmasked arcmin² (NaN where less than 25% of a bin is usable)."""
    h = binCounts(cat["x"], cat["y"])
    with np.errstate(divide="ignore", invalid="ignore"):
        d = h / areaArcmin2(run, GOOD_FRAC * DENSITY_BIN_PX ** 2)
    d[GOOD_FRAC < 0.25] = np.nan
    return d, int(h.sum())


def gaiaCompletenessMap(r, minStars=3):
    """Fraction of the Gaia stars in each bin matched by the run (NaN with < minStars Gaia stars)."""
    _, ig, _ = gaiaMatch(r)
    nAll = binCounts(gaia["x"], gaia["y"])
    nHit = binCounts(gaia["x"][ig], gaia["y"][ig])
    with np.errstate(divide="ignore", invalid="ignore"):
        c = nHit / nAll
    c[(nAll < minStars) | (GOOD_FRAC < 0.25)] = np.nan
    return c


haveGaia = gaia is not None and len(gaia)
maps = [(r["name"], *densityMap(r["cat"])) for r in withCat]
if haveGaia:
    maps.append(("Gaia DR3", *densityMap(gaia)))
vmax = max(np.nanpercentile(m, 99.5) for _, m, _ in maps)
diffs = [(name, m - maps[0][1]) for name, m, _ in maps[1:len(withCat)]]      # runs only, not Gaia
comps = [(r["name"], gaiaCompletenessMap(r)) for r in withCat] if haveGaia else []
panels = len(maps) + 1 + len(diffs) + len(comps)
ncol = min(panels, 3)
nrow = int(np.ceil(panels / ncol))
fig, axs = plt.subplots(nrow, ncol, figsize=(6.2 * ncol, 5.6 * nrow), squeeze=False)
axs = list(axs.ravel())
for name, m, n in maps:
    ax = axs.pop(0)
    h = ax.imshow(m, extent=DENS_EXT, cmap=SEQMAP, vmin=0, vmax=vmax)
    plt.colorbar(h, ax=ax, fraction=0.046, label="sources / unmasked arcmin²")
    ax.set_title(f"{name}: source density\n{n} sources, median {np.nanmedian(m):.0f} / arcmin²")
with np.errstate(divide="ignore", invalid="ignore"):
    maskedFrac = np.where(COV_FRAC > 0, 1 - GOOD_FRAC / COV_FRAC, np.nan)
ax = axs.pop(0)
h = ax.imshow(maskedFrac, extent=DENS_EXT, cmap=SEQMAP, vmin=0, vmax=1)
plt.colorbar(h, ax=ax, fraction=0.046, label="masked fraction")
ax.set_title(f"{run['name']}: masked fraction of each bin\n(> 75% masked: blank in the other maps)")
for name, d in diffs:
    ax = axs.pop(0)
    v = np.nanmax(np.abs(d)) if np.isfinite(d).any() else 1.0
    h = ax.imshow(d, extent=DENS_EXT, cmap=DIVERGING, vmin=-v, vmax=v)
    plt.colorbar(h, ax=ax, fraction=0.046, label="Δ sources / unmasked arcmin²")
    ax.set_title(f"{name} − {maps[0][0]}\n(red: {name} finds more; blue: fewer)")
for name, c in comps:
    ax = axs.pop(0)
    h = ax.imshow(c, extent=DENS_EXT, cmap=SEQMAP, vmin=0, vmax=1)
    plt.colorbar(h, ax=ax, fraction=0.046, label="fraction of Gaia stars matched")
    ax.set_title(f"{name}: Gaia completeness per bin\n(median {np.nanmedian(c):.2f}; blank: < 3 Gaia stars)")
for ax in axs:
    ax.set_axis_off()
show(fig, "density")

## 8. Extras

### Luminosity function and nearest-neighbour separations
The luminosity function shows where each run's detection runs out (the turnover) and spurious
excesses. The nearest-neighbour separation shows how close together each run resolves sources (the
dashed line is `minCentroidSeparation`); a pile-up at the cut, or a lack of pairs below ~1 FWHM,
point to deblending limits.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))
allMag = np.concatenate([np.asarray(r["cat"]["mag"], float) for r in withCat])
allMag = allMag[np.isfinite(allMag)]
mb = np.arange(np.floor(np.percentile(allMag, 0.5)), np.ceil(np.percentile(allMag, 99.9)) + 0.25, 0.25)
for i, r in enumerate(withCat):
    axs[0].hist(r["cat"]["mag"], bins=mb, histtype="step", color=RUN_COLORS[i % 6], lw=1.5,
                label=f"{r['name']} ({np.sum(np.isfinite(r['cat']['mag']))})")
    xy = np.c_[r["cat"]["x"], r["cat"]["y"]]
    d, _ = cKDTree(xy).query(xy, k=2)
    axs[1].hist(d[:, 1], bins=np.logspace(-1, 2, 80), histtype="step", color=RUN_COLORS[i % 6],
                lw=1.5, label=r["name"])
if gaia is not None and len(gaia):
    axs[0].hist(gaia["G"], bins=mb, histtype="stepfilled", color=BLUE, alpha=0.25,
                label="Gaia G (other band)", zorder=0)
axs[0].set_yscale("log")
axs[0].set_xlabel("mag")
axs[0].set_title("luminosity function")
axs[0].legend(fontsize=8)
axs[1].set_xscale("log")
axs[1].axvline(MIN_CENTROID_SEPARATION, color=INK, ls="--", lw=0.8)
if run.get("psf") is not None:
    axs[1].axvline(run["psf"]["fwhmCenter"], color=INK, ls=":", lw=1, label="PSF FWHM")
axs[1].set_xlabel("nearest-neighbour separation [px]")
axs[1].set_title("nearest-neighbour separations")
axs[1].legend(fontsize=8)
show(fig, "lf_nn")

### Run-to-run comparison
Each comparison run matched to the main run within `MATCH_RADIUS_PX`: magnitude differences, position
differences, and where the sources found by only one of the two runs are (and how bright they are).

In [ ]:
def compareRuns(a, b):
    ia, ib, d = matchXY(a["cat"]["x"], a["cat"]["y"], b["cat"]["x"], b["cat"]["y"], MATCH_RADIUS_PX)
    onlyA = np.setdiff1d(np.arange(len(a["cat"])), ia)
    onlyB = np.setdiff1d(np.arange(len(b["cat"])), ib)
    print(f"{a['name']} vs {b['name']}: {len(ia)} matched, {len(onlyA)} only in {a['name']}, "
          f"{len(onlyB)} only in {b['name']}")
    fig, axs = plt.subplots(1, 3, figsize=(19, 5.5))
    ma, mbm = np.asarray(a["cat"]["mag"][ia], float), np.asarray(b["cat"]["mag"][ib], float)
    dm = mbm - ma
    axs[0].plot(ma, dm, ".", ms=2.5, alpha=0.45, color=BLUE)
    c, m, lo, hi = runningMedian(ma, dm, np.arange(np.nanmin(ma), np.nanmax(ma) + 0.5, 0.5))
    axs[0].plot(c, m, "-", color=RED)
    axs[0].fill_between(c, lo, hi, color=RED, alpha=0.15, lw=0)
    axs[0].axhline(0, color=INK, lw=0.8)
    axs[0].set_ylim(-1, 1)
    axs[0].set_xlabel(f"mag ({a['name']})")
    axs[0].set_ylabel(f"mag {b['name']} − {a['name']}")
    axs[0].set_title(f"matched: {len(ia)}, median Δmag {np.nanmedian(dm):+.3f}")
    dx = b["cat"]["x"][ib] - a["cat"]["x"][ia]
    dy = b["cat"]["y"][ib] - a["cat"]["y"][ia]
    axs[1].plot(dx, dy, ".", ms=2.5, alpha=0.45, color=BLUE)
    axs[1].set_xlim(-MATCH_RADIUS_PX, MATCH_RADIUS_PX)
    axs[1].set_ylim(-MATCH_RADIUS_PX, MATCH_RADIUS_PX)
    axs[1].set_aspect("equal")
    axs[1].set_xlabel("Δx [px]")
    axs[1].set_ylabel("Δy [px]")
    axs[1].set_title(f"position differences, median ({np.median(dx):+.3f}, {np.median(dy):+.3f}) px")
    bins = np.arange(np.floor(np.nanpercentile(allMag, 0.5)), np.ceil(np.nanpercentile(allMag, 99.9)) + 0.25, 0.25)
    axs[2].hist(a["cat"]["mag"][onlyA], bins=bins, histtype="step", lw=1.8, color=RUN_COLORS[0],
                label=f"only in {a['name']}")
    axs[2].hist(b["cat"]["mag"][onlyB], bins=bins, histtype="step", lw=1.8, color=RUN_COLORS[1],
                label=f"only in {b['name']}")
    axs[2].set_yscale("log")
    axs[2].set_xlabel("mag")
    axs[2].set_title("unmatched sources")
    axs[2].legend(fontsize=8)
    show(fig, f"compare_{a['name']}_{b['name']}")

    im, ext = cut(a, masked(a, "data"), ZOOM)
    if im is not None:
        ca, cb = a["cat"][onlyA], b["cat"][onlyB]
        za, zb = inZoom(ca, ZOOM), inZoom(cb, ZOOM)
        fig, ax = plt.subplots(figsize=(9, 9))
        h = ax.imshow(im, extent=ext, norm=imageNorm(im), cmap=GRAY)
        plt.colorbar(h, ax=ax, fraction=0.046, pad=0.02, label=FLUX_LABEL)
        both = inZoom(a["cat"][ia], ZOOM)
        ax.plot(a["cat"]["x"][ia][both], a["cat"]["y"][ia][both], **MK_BOTH, label=f"in both ({both.sum()})")
        ax.plot(ca["x"][za], ca["y"][za], **MK_ONLY_A, label=f"only {a['name']} ({za.sum()})")
        ax.plot(cb["x"][zb], cb["y"][zb], **MK_ONLY_B, label=f"only {b['name']} ({zb.sum()})")
        ax.set_xlim(ext[0], ext[1])
        ax.set_ylim(ext[2], ext[3])
        ax.legend(**LEGEND_ON_IMAGE)
        ax.set_title(f"{a['name']} vs {b['name']} in the zoom box")
        show(fig, f"compare_zoom_{a['name']}_{b['name']}")


for b in withCat[1:]:
    compareRuns(run, b)
if len(withCat) == 1:
    print("No comparison runs: set COMPARE_PIPE_CROWD and/or MEAS_CROWD_DIRS.")

### Summary table

`step time` is the sum of the timed steps (only for runs made with the step timers); `quantum time`
is pipetask's wall time for the whole quantum (available for every `pipe_crowd` run, including
reading and writing). `s / 1000 src` uses the step time when there is one, else the quantum time
(`time from` says which). The Gaia columns count the Gaia DR3 stars on the detector, how many the
run matched within `MATCH_RADIUS_PX`, and how many it missed.

In [ ]:
rows = []
for r in runs:
    t = r.get("timing")
    stepT = float(np.sum(leaf(t)["seconds"])) if hasSteps(r) else np.nan
    q = t[t["label"] == "pipetask: quantum total"] if t is not None and len(t) else []
    quantT = float(q["seconds"][0]) if len(q) else np.nan
    perT, perFrom = runTime(r)
    peak = float(t["peakRssMB"][0]) / 1e3 if t is not None and len(t) and "peakRssMB" in t.colnames \
        and t["peakRssMB"][0] > 0 else np.nan
    chi = chiImage(r) if r.get("resid") is not None else None
    v = chi[np.isfinite(chi)] if chi is not None else np.array([])
    nCat = len(r["cat"]) if r.get("cat") is not None else -1
    nG = nGm = nGu = -1
    if r.get("cat") is not None and gaia is not None and len(gaia):
        _, ig, _ = gaiaMatch(r)
        nG, nGm = len(gaia), len(ig)
        nGu = nG - nGm
    rows.append((r["name"], nCat, nG, nGm, nGu, stepT, quantT,
                 1e3 * perT / nCat if nCat > 0 and np.isfinite(perT) else np.nan, perFrom, peak,
                 1.4826 * np.median(np.abs(v - np.median(v))) if v.size else np.nan,
                 np.mean(np.abs(v) > 5) if v.size else np.nan))
summary = Table(rows=rows, names=["run", "N sources", "N Gaia", "N Gaia matched", "N Gaia unmatched",
                                  "step time [s]", "quantum time [s]", "s / 1000 src", "time from",
                                  "peak memory [GB]", "σ_MAD χ", "frac |χ|>5"])
if gaiaSummary is not None:
    for c in ("compl. G<19", "compl. G<20", "σ_x [mas]", "σ_y [mas]"):
        summary[c] = [gaiaSummary[c][list(gaiaSummary["run"]).index(n)] if n in gaiaSummary["run"] else np.nan
                      for n in summary["run"]]
for c in summary.colnames:
    if summary[c].dtype.kind == "f":
        summary[c].format = ".3f"
summary

### Saving the catalogues for later comparison
Writes the main run's catalogue (with the Gaia match) and its timing table, so other notebooks can
compare against this run without the Butler.

In [ ]:
OUT_DIR = SAVE_DIR or "."
cat = run["cat"].copy()
if gaia is not None and len(gaia):
    ia, ig, d = gaiaMatch(run)
    cat["gaiaG"] = np.full(len(cat), np.nan)
    cat["gaiaSepPx"] = np.full(len(cat), np.nan)
    cat["gaiaG"][ia] = gaia["G"][ig]
    cat["gaiaSepPx"][ia] = d
base = os.path.join(OUT_DIR, f"{run['name']}_v{VISIT}_d{DETECTOR}")
cat.write(base + "_catalog.fits", overwrite=True)
if run.get("timing") is not None:
    run["timing"].write(base + "_timing.csv", overwrite=True)
print("wrote", base + "_catalog.fits", "and _timing.csv" if run.get("timing") is not None else "")

## 9. Cutouts at the corners and along the diagonal

`CUTOUT_SIZE`-pixel boxes at the four detector corners and `N_DIAGONAL` boxes evenly spaced along the
diagonal from the bottom-left to the top-right corner. Corners are where the PSF and the WCS are
least well constrained and vignetting / edge effects show up; the diagonal samples the detector
in between, so a trend from one corner to the other is easy to spot.

For each box, two figures: first the images on their own, then the same images with the sources
and Gaia stars marked. Each has one row per run with images (so runs can be compared pixel by
pixel), with the data (overlay: red ○ = sources, blue + = Gaia matched, blue ◆ = Gaia missed), the
model, and the residual (`CUTOUT_RESIDUAL_STYLE`; overlay: dark ○ = sources). After the images, a table and a plot of the numbers
per box: source density, Gaia completeness and residual χ statistics.

In [ ]:
def cutoutBoxes(r, size=CUTOUT_SIZE, nDiag=N_DIAGONAL):
    """(code, description, (x0, y0, size)) for the four corners and nDiag boxes along the diagonal."""
    ny, nx = r["shape"]
    size = int(min(size, nx, ny))
    xa, ya = r["x0"], r["y0"]
    xb, yb = r["x0"] + nx - size, r["y0"] + ny - size
    boxes = [("BL", "corner bottom-left", (xa, ya, size)), ("BR", "corner bottom-right", (xb, ya, size)),
             ("TL", "corner top-left", (xa, yb, size)), ("TR", "corner top-right", (xb, yb, size))]
    for i, t in enumerate(np.linspace(0, 1, nDiag + 2)[1:-1]):
        boxes.append((f"D{i + 1}", f"diagonal {i + 1} of {nDiag}",
                      (int(round(xa + t * (nx - size))), int(round(ya + t * (ny - size))), size)))
    return boxes


BOXES = cutoutBoxes(run)
imgRuns = [r for r in runs if r.get("data") is not None and r.get("cat") is not None]
gaiaMatches = {r["name"]: gaiaMatch(r) for r in imgRuns} if gaia is not None and len(gaia) else {}

# overview: where the boxes are
img, ext, f = fullView(run, masked(run, "data"))
fig, ax = plt.subplots(figsize=(9, 9 * run["shape"][0] / run["shape"][1] + 0.6))
h = ax.imshow(img, extent=ext, norm=imageNorm(img), cmap=GRAY)
plt.colorbar(h, ax=ax, fraction=0.046, pad=0.02, label=FLUX_LABEL)
for code, desc, (bx, by, bs) in BOXES:
    ax.add_patch(Rectangle((bx - 0.5, by - 0.5), bs, bs, fill=False, ec=RED if code[0] != "D" else BLUE,
                           lw=1.8, path_effects=EDGE_WHITE))
    ax.text(bx + bs / 2, by + bs / 2, code, ha="center", va="center", fontsize=11, color="white",
            path_effects=[pe.withStroke(linewidth=2.6, foreground=RED if code[0] != "D" else BLUE)])
ax.set_title(f"{run['name']}: cutout positions ({BOXES[0][2][2]} px boxes; red = corners, blue = diagonal)")
show(fig, "cutouts_overview")


def plotCutout(code, desc, box, overlay=True):
    fig, axs = plt.subplots(len(imgRuns), 3, figsize=(16, 5.4 * len(imgRuns)), squeeze=False)
    for row, r in zip(axs, imgRuns):
        ims = [cut(r, masked(r, k), box) for k in ("data", "model", "resid")]
        if ims[0][0] is None:
            for ax in row:
                ax.set_axis_off()
            row[0].set_title(f"{r['name']}: no image here")
            continue
        norm = imageNorm(ims[0][0])
        sel = inZoom(r["cat"], box)
        cat = r["cat"][sel]
        for ax, (im, ext), lab, kw in zip(row, ims, ["data", "model", "residual (data − model)"],
                                          [dict(norm=norm, cmap=GRAY), dict(norm=norm, cmap=GRAY),
                                           residualStyle(ims[2][0], CUTOUT_RESIDUAL_STYLE)]):
            h = ax.imshow(im, extent=ext, **kw)
            plt.colorbar(h, ax=ax, fraction=0.046, pad=0.02, label=FLUX_LABEL)
            if overlay:
                onDiv = lab.startswith("residual") and CUTOUT_RESIDUAL_STYLE != "gray"
                ax.plot(cat["x"], cat["y"], **(MK_SRC_ON_DIVERGING if onDiv else MK_SRC),
                        label=f"{r['name']} sources ({len(cat)})")
            ax.set_xlim(ext[0], ext[1])
            ax.set_ylim(ext[2], ext[3])
            ax.set_title(f"{r['name']}: {lab}" + ("" if overlay else f" ({len(cat)} sources)")
                         * (lab == "data"))
        if not overlay:
            continue
        if r["name"] in gaiaMatches:
            _, ig, _ = gaiaMatches[r["name"]]
            matched = np.zeros(len(gaia), bool)
            matched[ig] = True
            gz = inZoom(gaia, box)
            row[0].plot(gaia["x"][gz & matched], gaia["y"][gz & matched], **MK_GAIA,
                        label=f"Gaia matched ({(gz & matched).sum()})")
            row[0].plot(gaia["x"][gz & ~matched], gaia["y"][gz & ~matched], **MK_GAIA_MISSED,
                        label=f"Gaia missed ({(gz & ~matched).sum()})")
        row[0].legend(**LEGEND_ON_IMAGE)
    bx, by, bs = box
    fig.suptitle(f"{code}: {desc} — x {bx}…{bx + bs - 1}, y {by}…{by + bs - 1}"
                 + (" — sources and Gaia overlaid" if overlay else " — images only"))
    show(fig, f"cutout_{code}_{'overlay' if overlay else 'plain'}")


def cutoutStats(r, box):
    """Numbers for one run in one box."""
    sel = inZoom(r["cat"], box)
    bad = cut(r, r["bad"].astype(float), box)[0] if r.get("bad") is not None else None
    good = (bad.size - bad.sum()) if bad is not None else box[2] ** 2
    chi, _ = cut(r, chiImage(r), box) if r.get("resid") is not None else (None, None)
    v = chi[np.isfinite(chi)] if chi is not None else np.array([])
    out = dict(run=r["name"], N=int(sel.sum()), density=sel.sum() / max(areaArcmin2(r, good), 1e-9),
               chiMad=1.4826 * np.median(np.abs(v - np.median(v))) if v.size else np.nan,
               fracChi5=np.mean(np.abs(v) > 5) if v.size else np.nan, gaiaN=0, gaiaComp=np.nan)
    if r["name"] in gaiaMatches:
        _, ig, _ = gaiaMatches[r["name"]]
        gz = inZoom(gaia, box)
        matched = np.zeros(len(gaia), bool)
        matched[ig] = True
        out["gaiaN"] = int(gz.sum())
        out["gaiaComp"] = (gz & matched).sum() / gz.sum() if gz.sum() else np.nan
    return out


for code, desc, box in BOXES:
    plotCutout(code, desc, box, overlay=False)   # the images on their own
    plotCutout(code, desc, box, overlay=True)    # the same with sources and Gaia stars marked

rows = []
for code, desc, box in BOXES:
    for r in imgRuns:
        st = cutoutStats(r, box)
        rows.append(dict(cutout=code, x0=box[0], y0=box[1], **st))
cutoutTable = Table(rows=[list(rw.values()) for rw in rows], names=list(rows[0].keys()))
cutoutTable.rename_columns(["density", "chiMad", "fracChi5", "gaiaN", "gaiaComp"],
                           ["sources / arcmin²", "σ_MAD χ", "frac |χ|>5", "N Gaia", "Gaia compl."])
for c in ("sources / arcmin²", "σ_MAD χ", "frac |χ|>5", "Gaia compl."):
    cutoutTable[c].format = ".3f"

codes = [b[0] for b in BOXES]
fig, axs = plt.subplots(1, 3, figsize=(17, 4.6))
for i, r in enumerate(imgRuns):
    t = cutoutTable[cutoutTable["run"] == r["name"]]
    for ax, col in zip(axs, ("sources / arcmin²", "Gaia compl.", "σ_MAD χ")):
        ax.plot(range(len(codes)), t[col], "-o", color=RUN_COLORS[i % 6], label=r["name"])
for ax, col, note in zip(axs, ("sources / arcmin²", "Gaia compl.", "σ_MAD χ"),
                         ("", "1 = every Gaia star found", "1 = residuals consistent with noise")):
    ax.set_xticks(range(len(codes)), codes)
    ax.axvspan(-0.5, 3.5, color="#eeeeee", zorder=0)          # corners
    ax.set_title(col + (f"\n({note})" if note else ""))
    ax.set_xlabel("cutout (grey: corners; D1 to D%d: bottom-left to top-right)" % N_DIAGONAL)
axs[2].axhline(1, color=INK, ls="--", lw=0.8)
axs[1].set_ylim(-0.02, 1.05)
axs[0].legend()
show(fig, "cutouts_stats")
cutoutTable